# Bee Haven: Weather data from the BrightSky API ➜ Bronze & Silver

The hive sensors tell us what happens **inside** the hive. To understand *why* bees fly more on some days than others,
we need what happens **outside**: temperature, sunshine, rain and wind. This notebook:

1. finds the date range covered by our cleaned hive data (silver),
2. asks the **BrightSky weather API** for the hourly weather at the Schwartau hive for exactly that range,
3. saves the **raw JSON** answer untouched in the **bronze** layer,
4. flattens and cleans it (same LMS checklist as the sensor data) and saves it as **Parquet in silver**,
5. makes a few charts to check the result and give a first look at weather vs bee activity.

### Wait: do we need an API key?
**No.** Many APIs (Google Maps, OpenWeather …) require an **API key**: a secret password-like string sent with each request, so the provider knows
*who* is asking. They use it to count your usage, bill you, or block abuse. BrightSky is a free, public service built on open data from the
German weather service (DWD), so it doesn't need to know who you are. We just send a request.
(If you ever use an API that *does* need a key: never write it in the notebook. Put it in an environment variable or Azure Key Vault, like the storage account key.)

### Vocabulary in one minute
| Term | Meaning here |
|---|---|
| **API** | A "menu" of data a service offers to programs. BrightSky's menu item is `/weather` |
| **Endpoint / URL** | Where we send the order: `https://api.brightsky.dev/weather` |
| **Parameters** | The details of the order: location (`lat`, `lon`) and time range (`date`, `last_date`) |
| **Request** | Our program sending the order (`requests.get(...)`) |
| **Status code** | The reply's first word: `200` = OK, `400` = bad request, `404` = not found, `429` = too many requests, `5xx` = server problem |
| **JSON** | The format of the answer: *semi-structured* nested `{key: value}` objects and `[lists]` |

### Charts in this notebook
| Question | Chart |
|---|---|
| How much of each weather measure is missing? | Bar chart |
| Which weather station supplied the data? | Bar chart |
| How does outside temperature compare to hive temperature over time? | Line chart |
| How does rainfall vary by month? | Bar chart |
| Which weather conditions occur, and how often? | Bar chart |
| Does weather drive bee traffic? | Scatter plots |

## 0. Setup
Same folder layout as the bronze→silver notebook:
```
Bee Haven- Azure/
├── bronze/
│   ├── new/                  <- raw hive CSVs
│   └── archive/weather/      <- raw API answers (JSON) are saved here
├── silver/                   <- flow/, humidity/, temperature/, weight/ … and now weather/
└── notebooks/
    ├── 01_bronze_to_silver_local.ipynb
    └── 02_weather_api_to_silver.ipynb   <- this notebook
```
`requests` is the standard Python library for talking to web APIs. If it's missing: `%pip install requests`.

In [ ]:
import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

ROOT = Path('..')
SILVER = ROOT / 'silver'
BRONZE_WEATHER = ROOT / 'bronze' / 'archive' / 'weather'
PLACE = 'schwartau'

# Where the hives are. A dictionary per location, so adding Würzburg later is one more line.
COORDS = {
    'schwartau': {'lat': 53.919444, 'lon': 10.6975},
    'wurzburg':  {'lat': 49.783333, 'lon': 9.933333},
}

URL = 'https://api.brightsky.dev/weather'
HEADERS = {'Accept': 'application/json'}   # "please answer in JSON"

# same chart style as notebook 01
BLUE, ORANGE, AQUA, RED, GREY = '#2a78d6', '#eb6834', '#1baf7a', '#d03b3b', '#898781'
plt.rcParams.update({
    'figure.figsize': (12, 4), 'figure.dpi': 100,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'grid.linewidth': 0.6,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'axes.labelcolor': '#52514e', 'xtick.color': '#52514e', 'ytick.color': '#52514e',
    'lines.linewidth': 1.2, 'legend.frameon': False,
})

## 1. Which dates do we need?
**Why:** we only want weather for the period our hive sensors cover. Asking for more wastes time; asking for less leaves hive readings without weather.
We read the **newest** silver file of each sensor (file names contain the run time, so sorting the names puts the newest last)
and take the earliest and latest timestamp across all four.

In [ ]:
def latest_silver(sensor):
    files = sorted((SILVER / sensor).glob(f'{PLACE}__*.parquet'))
    if not files:
        raise FileNotFoundError(f'No silver file for {sensor}. Run notebook 01 first.')
    return files[-1]

hive = {s: pd.read_parquet(latest_silver(s)) for s in ['flow', 'humidity', 'temperature', 'weight']}
for s, df in hive.items():
    print(f"{s:12s} {latest_silver(s).name}   {df['timestamp'].min()}  →  {df['timestamp'].max()}")

start_date = min(df['timestamp'].min() for df in hive.values())
end_date   = max(df['timestamp'].max() for df in hive.values())
print('\nweather needed from', start_date, 'to', end_date)

## 2. One small request first
**Why start small:** a one-day request answers in a second and is easy to read. We make sure it works and understand the answer
*before* asking for 2.5 years of hourly data.

The parameters are the "order":
- `lat`, `lon`: the hive location. BrightSky finds the nearest weather stations itself.
- `date`, `last_date`: the time window (ISO format; a timezone of `+00:00` means UTC).
- We leave `units` at its default (`dwd`, the German weather service's units): °C, mm of rain, km/h wind, hPa pressure, minutes of sunshine per hour.

In [ ]:
params = COORDS[PLACE] | {'date': start_date.isoformat(), 'last_date': (start_date + pd.Timedelta(days=1)).isoformat()}
response = requests.get(URL, headers=HEADERS, params=params, timeout=60)

print('status code:', response.status_code)      # 200 = OK
print('the full URL that was called:', response.url)

The printed URL is exactly what you could paste into a browser. The API is "just" a web address with the order written into it.
`requests` builds it from the dictionary for us and handles special characters like `+` and `:`.

## 3. Explore the JSON answer
`response.json()` turns the JSON text into Python **dictionaries and lists**. This is *semi-structured* data:
it has structure, but it's nested, and not every record has the same keys (as we'll see with `fallback_source_ids`).

In [ ]:
sample = response.json()
print('top-level keys:', list(sample.keys()))
print('number of hourly records:', len(sample['weather']))
print('number of weather stations used:', len(sample['sources']))

In [ ]:
# one hourly record, pretty-printed: note the nested dictionary inside it
print(json.dumps(sample['weather'][0], indent=2))

In [ ]:
# the stations ("sources") the data came from
pd.DataFrame(sample['sources'])[['id', 'station_name', 'distance', 'height', 'lat', 'lon']]

**Reading the structure:**
- `"weather"` is a **list** with one **object** per hour: timestamp, temperature, precipitation …
- `"sources"` lists the weather **stations**. Each hourly record has a `source_id` pointing to its main station.
- `"fallback_source_ids"` is a **nested object**. When the main (closest) station didn't measure something, BrightSky filled it from another,
  more distant station and notes *which* one per measure. So one row can mix several stations. Our solution: for each measure we record **how far away** it was measured (`<measure>_source_distance`, in metres).
  Analysts can then decide, e.g., to trust sunshine measured 2 km away more than sunshine from 30 km away.

`pd.json_normalize` shows what happens if we flatten it naively: nested keys become dotted column names.

In [ ]:
pd.json_normalize(sample['weather']).filter(like='fallback').head(3)

## 4. Fetch the whole period, month by month → bronze
**Why month by month (instead of one huge request)?**
- One 2.5-year request returns ~21,000 hours in one big answer. If it times out, everything is lost and you start again.
- Monthly chunks are small and fast. If one fails, we **retry** just that month.
- Each month's raw answer is saved as its own JSON file in **bronze**, so re-running later only needs to download what's missing.

**Why save the raw JSON at all?** Bronze keeps the data *exactly as received*. If we find a bug in our cleaning later,
we can re-process from these files without calling the API again, and we can prove what the API returned on the day we asked.

`time.sleep(0.5)` pauses briefly between requests. A free public API deserves polite use (and too many quick requests can get a `429` reply).

In [ ]:
BRONZE_WEATHER.mkdir(parents=True, exist_ok=True)

def fetch_period(place, start, end, retries=3):
    """Ask BrightSky for one period. Returns the parsed JSON, or raises after `retries` failures."""
    params = COORDS[place] | {'date': start.isoformat(), 'last_date': end.isoformat()}
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(URL, headers=HEADERS, params=params, timeout=60)
            if r.status_code == 200:
                return r.json()
            print(f'  attempt {attempt}: status {r.status_code}: {r.text[:120]}')
        except requests.RequestException as e:          # no internet, timeout, ...
            print(f'  attempt {attempt}: {type(e).__name__}: {e}')
        time.sleep(2 * attempt)                          # wait a bit longer after each failure
    raise RuntimeError(f'Could not fetch weather for {start} → {end}')

# month boundaries covering start_date..end_date
edges = list(pd.date_range(start_date.floor('D'), end_date.ceil('D'), freq='MS', tz='UTC'))
edges = sorted({start_date.floor('h'), *edges, end_date.ceil('h')})
edges = [e for e in edges if start_date.floor('h') <= e <= end_date.ceil('h')]

raw_files = []
for chunk_start, chunk_end in zip(edges[:-1], edges[1:]):
    path = BRONZE_WEATHER / f"{PLACE}__{chunk_start:%Y-%m-%dT%H}_{chunk_end:%Y-%m-%dT%H}.json"
    if not path.exists():                                # already downloaded? then skip
        data = fetch_period(PLACE, chunk_start, chunk_end)
        path.write_text(json.dumps(data, indent=2))
        time.sleep(0.5)
    raw_files.append(path)

print(len(raw_files), 'monthly JSON files in', BRONZE_WEATHER)

## 5. Flatten the JSON into a table
For every hourly record we:
1. keep the actual measurements,
2. add a `<measure>_source_distance` column: the distance from the hive to the station that measured it (main station, or the fallback station if there is one),
3. drop fields that aren't observations: `icon` (a picture name), `precipitation_probability*` (only used for *forecasts*; we use historical data),
   `source_id` / `fallback_source_ids` (replaced by the distances), and `visibility` (not relevant for bees).

We **keep `condition`** (dry / rain / snow …) as a categorical column. It's useful, and it's the categorical-consistency check the LMS asks for.
Wrapped in a function, so it runs for each monthly file.

In [ ]:
NOT_MEASURES = {'timestamp', 'source_id', 'fallback_source_ids', 'condition', 'icon',
                'precipitation_probability', 'precipitation_probability_6h'}
DROP_COLUMNS = ['source_id', 'fallback_source_ids', 'icon', 'visibility',
                'precipitation_probability', 'precipitation_probability_6h']

def flatten(weather_json):
    distance = {s['id']: s['distance'] for s in weather_json['sources']}   # station id -> metres from hive
    rows = []
    for record in weather_json['weather']:
        row = dict(record)
        fallback = record.get('fallback_source_ids') or {}
        for measure in record:
            if measure in NOT_MEASURES:
                continue
            station = fallback.get(measure, record['source_id'])
            row[f'{measure}_source_distance'] = distance.get(station)
        rows.append(row)
    return pd.DataFrame(rows).drop(columns=DROP_COLUMNS, errors='ignore')

weather_raw = pd.concat([flatten(json.loads(p.read_text())) for p in raw_files], ignore_index=True)
print(weather_raw.shape)
weather_raw.head()

## 6. Clean: the same LMS checklist as the hive data
### 6.1 Column names
BrightSky already uses `snake_case`, but we apply the same rule as for the sensor files so every silver table follows one convention.

In [ ]:
def standardise_columns(df):
    df = df.copy()
    df.columns = [re.sub(r'_+', '_', re.sub(r'[^0-9a-z]+', '_', c.strip().lower())).strip('_') for c in df.columns]
    return df

weather = standardise_columns(weather_raw)
list(weather.columns)

### 6.2 Timestamps → UTC, duplicates, regularity
The API sends timestamps as text with an offset (`2017-01-01T12:00:00+00:00`). `utc=True` makes them proper UTC datetimes,
the same as the hive data, so they can be joined later.
Monthly chunks share their boundary hour (the last hour of one month is the first of the next request), so we expect a few
**duplicate timestamps**. We drop those, then check that we have one row per hour.

In [ ]:
weather['timestamp'] = pd.to_datetime(weather['timestamp'], utc=True)
print('exact duplicate rows:     ', weather.duplicated().sum())
print('duplicate timestamps:     ', weather['timestamp'].duplicated().sum())
weather = weather.drop_duplicates(subset='timestamp', keep='first').sort_values('timestamp').reset_index(drop=True)

steps = weather['timestamp'].diff().value_counts()
print('\ntime between rows:'); print(steps.head())
expected = pd.date_range(weather['timestamp'].min(), weather['timestamp'].max(), freq='h')
print('\nmissing hours:', len(expected.difference(weather['timestamp'])))

### 6.3 Data types
Measurements must be numbers. Some columns arrive as whole numbers (e.g. `relative_humidity`), but they may contain gaps,
so we make them all `float` for consistency. `condition` becomes a **category**: it stores each label once, which saves memory and makes its allowed values explicit.

In [ ]:
measure_cols = [c for c in weather.columns if c not in ('timestamp', 'condition')]
weather[measure_cols] = weather[measure_cols].apply(pd.to_numeric, errors='coerce').astype('float64')
weather.dtypes.value_counts()

### 6.4 Categorical consistency: `condition`
**Why:** a category spelled three ways (`Rain`, `rain `, `RAIN`) becomes three separate groups in every chart and group-by.
We normalise it (trim spaces, lowercase) and check that only the values BrightSky documents appear.

In [ ]:
ALLOWED_CONDITIONS = ['dry', 'fog', 'rain', 'sleet', 'snow', 'hail', 'thunderstorm']
weather['condition'] = weather['condition'].astype('string').str.strip().str.lower()
unexpected = set(weather['condition'].dropna().unique()) - set(ALLOWED_CONDITIONS)
print('unexpected condition labels:', unexpected or 'none')
weather['condition'] = pd.Categorical(weather['condition'], categories=ALLOWED_CONDITIONS)

**Bar chart: how often does each condition occur?** Conditions are categories, so a bar chart compares their counts.
The y-axis is logarithmic because "dry" is far more common than everything else; on a normal axis the rare conditions would be invisible.

In [ ]:
counts = weather['condition'].cat.add_categories('missing').fillna('missing').value_counts()
counts = counts[counts > 0]          # hide conditions that never occurred
fig, ax = plt.subplots(figsize=(10, 3.5))
bars = ax.bar(counts.index.astype(str), counts.values, color=[GREY if i == 'missing' else BLUE for i in counts.index.astype(str)], width=0.6)
ax.bar_label(bars, labels=[f'{v:,}' for v in counts.values], padding=3, color='#52514e')
ax.set_yscale('log'); ax.set_ylabel('hours (log scale)'); ax.set_title('Weather condition per hour')
ax.grid(axis='x', visible=False)
plt.show()

### 6.5 Impossible values (plausibility ranges)
A weather station can glitch just like a hive sensor. We check each measure against physically possible ranges for northern Germany.
Anything outside becomes `NaN`, the same principle as notebook 01: fix what's provably wrong, don't invent values.

In [ ]:
PLAUSIBLE = {
    'temperature': (-30, 40),        # °C
    'dew_point': (-40, 30),          # °C
    'relative_humidity': (0, 100),   # %
    'cloud_cover': (0, 100),         # %
    'precipitation': (0, 100),       # mm per hour
    'pressure_msl': (940, 1060),     # hPa
    'wind_speed': (0, 200),          # km/h
    'wind_gust_speed': (0, 250),     # km/h
    'wind_direction': (0, 360),      # degrees
    'wind_gust_direction': (0, 360), # degrees
    'sunshine': (0, 60),             # minutes of sunshine in the hour
    'solar': (0, 2),                 # kWh/m² in the hour
}
report = {}
for col, (lo, hi) in PLAUSIBLE.items():
    if col in weather:
        bad = weather[col].notna() & ~weather[col].between(lo, hi)
        report[col] = int(bad.sum())
        weather.loc[bad, col] = np.nan
pd.Series(report, name='values set to NaN')

### 6.6 Missing values
**Bar chart per column**, the same idea as in notebook 01. We keep missing values as `NaN`:
filling weather gaps is an analysis decision, made later.

In [ ]:
value_cols = [c for c in weather.columns if c != 'timestamp' and not c.endswith('_source_distance')]
missing = (weather[value_cols].isna().mean() * 100).sort_values()
fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.barh(missing.index, missing.values, color=[RED if v > 5 else BLUE for v in missing.values], height=0.6)
ax.bar_label(bars, labels=[f'{v:.1f} %' for v in missing.values], padding=3, color='#52514e')
ax.set_xlabel('% of hours missing'); ax.set_title('Missing weather values per measure (red = more than 5 %)')
ax.grid(axis='y', visible=False); ax.set_xlim(0, max(missing.max() * 1.25, 1))
plt.tight_layout(); plt.show()

Measures missing more often (typically `sunshine`) simply aren't recorded at every station for every hour.
The `_source_distance` columns tell the analyst how far away each value was measured.

### 6.7 Which station supplied the data?
**Bar chart:** for the main measures, what share of hours came from each station distance? Nearer is better.

In [ ]:
share = pd.DataFrame({
    m: weather[f'{m}_source_distance'].round(-2).div(1000).value_counts(normalize=True) * 100
    for m in ['temperature', 'precipitation', 'sunshine', 'wind_speed'] if f'{m}_source_distance' in weather
}).fillna(0).sort_index()
share.index = [f'{d:.1f} km' for d in share.index]

colors = [BLUE, AQUA, ORANGE, '#4a3aa7', '#eda100', GREY]
fig, ax = plt.subplots(figsize=(10, 3.8))
x = np.arange(len(share.columns)); width = 0.8 / len(share.index)
for i, (dist, row) in enumerate(share.iterrows()):
    ax.bar(x + i * width, row.values, width=width * 0.92, color=colors[i % len(colors)], label=f'station {dist} away')
ax.set_xticks(x + width * (len(share.index) - 1) / 2); ax.set_xticklabels(share.columns)
ax.set_ylabel('% of hours'); ax.set_title('Distance of the station each measure came from')
ax.legend(loc='upper right'); ax.grid(axis='x', visible=False)
plt.show()

## 7. Save to silver as Parquet
Same naming convention as the sensors: `silver/weather/schwartau__<run time>.parquet`.
These columns are what the Silver→Gold notebook uses (plus `condition`, which gold can use or ignore).

In [ ]:
assert str(weather['timestamp'].dt.tz) == 'UTC'
assert weather['timestamp'].is_unique and weather['timestamp'].is_monotonic_increasing
assert all(re.fullmatch(r'[a-z0-9_]+', c) for c in weather.columns)
for col, (lo, hi) in PLAUSIBLE.items():
    if col in weather:
        assert weather[col].dropna().between(lo, hi).all(), col
print('All checks passed ✅')

out_dir = SILVER / 'weather'
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"{PLACE}__{pd.Timestamp.now():%Y-%m-%dT%Hh%Mm%Ss}.parquet"
weather.to_parquet(out_path, index=False)
print(out_path, f'{out_path.stat().st_size/1e6:.2f} MB', weather.shape)

## 8. First look: weather vs the hive
Now we can put inside and outside side by side. The hourly weather and the minute-level hive data are brought to a **daily** level first
(in memory only; the proper join happens in the gold notebook).

**Line chart: outside vs hive temperature.** Both are °C, so they share one axis. This shows how the bees keep their brood nest
at ~35 °C in summer, no matter what the weather does.

In [ ]:
daily_weather = weather.set_index('timestamp').resample('D').agg({
    'temperature': 'mean', 'precipitation': 'sum', 'sunshine': 'sum'})
daily_weather['sunshine'] = daily_weather['sunshine'] / 60          # minutes → hours
daily_hive = pd.DataFrame({
    'hive_temp': hive['temperature'].set_index('timestamp')['temperature'].resample('D').mean(),
    'departures': hive['flow'].set_index('timestamp')['flow_out'].resample('D').sum().abs(),
})
daily_hive.loc[daily_hive['departures'] == 0, 'departures'] = np.nan
daily = daily_weather.join(daily_hive, how='inner')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily.index, daily['hive_temp'], color=ORANGE, label='inside the hive')
ax.plot(daily.index, daily['temperature'], color=BLUE, label='outside (weather station)')
ax.set_ylabel('°C (daily mean)'); ax.set_title('Inside vs outside temperature')
ax.legend(loc='upper left'); ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.show()

**Bar chart: rain per month.** Months are categories in time order, and a monthly total is a count-like amount, so bars suit it better than a line.

In [ ]:
monthly_rain = weather.set_index('timestamp')['precipitation'].resample('MS').sum()
fig, ax = plt.subplots(figsize=(12, 3.8))
ax.bar(monthly_rain.index, monthly_rain.values, width=20, color=BLUE)
ax.set_ylabel('mm'); ax.set_title('Total rainfall per month')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y')); ax.grid(axis='x', visible=False)
plt.show()

**Scatter plots: does weather drive bee traffic?** Each dot is one day. Outside temperature and sunshine have different units,
so each gets its own panel instead of sharing an axis.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
axes[0].scatter(daily['temperature'], daily['departures'], s=10, alpha=0.6, color=BLUE)
axes[0].set_xlabel('outside temperature, daily mean (°C)'); axes[0].set_ylabel('departures per day')
axes[0].set_title('Outside temperature vs bee departures')
axes[1].scatter(daily['sunshine'], daily['departures'], s=10, alpha=0.6, color=ORANGE)
axes[1].set_xlabel('sunshine (hours per day)'); axes[1].set_title('Sunshine vs bee departures')
plt.tight_layout(); plt.show()

print('correlation with daily departures:')
print(daily[['temperature', 'sunshine', 'precipitation']].corrwith(daily['departures']).round(2))

Look at the shape of each cloud and at the correlation numbers: which weather measure moves most with bee traffic?
These relationships are exactly what the gold layer is for: the Silver→Gold notebook joins the hive and weather tables on `timestamp`
into one analysis-ready dataset.

## Summary
| Step | What we did | Why |
|---|---|---|
| Date range | Min/max timestamp of the silver hive files | Request only the weather we need |
| Test request | One day first, check status code 200 | Cheap way to confirm the API call works |
| Explore JSON | `weather` list, `sources` list, nested `fallback_source_ids` | Understand semi-structured data before flattening it |
| Fetch | Monthly chunks, retries, polite pauses | Robust: a failure only costs one month |
| Bronze | Raw JSON saved per month | Reproducible; re-process without calling the API again |
| Flatten | One row per hour + distance per measure | Turns nested JSON into a table; records data quality per value |
| Clean | snake_case, UTC, duplicates, types, categories, ranges | Same LMS checklist as the hive data |
| Silver | Parquet in `silver/weather/` | Ready for the Silver→Gold join |